In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# =============================================================================
# Cell 1: Imports, paths, and global settings
# =============================================================================
import os, gc, shutil, time, json
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import pydicom
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# ---------- Paths ----------
DATA_DIR = Path("../input/competitions/rsna-knee-abnormality-detection")
TRAIN_CSV = DATA_DIR / "train.csv"
TRAIN_SERIES_CSV = DATA_DIR / "train_series.csv"
TEST_CSV = DATA_DIR / "test.csv"
TEST_SERIES_CSV = DATA_DIR / "test_series.csv"
TRAIN_DICOM_DIR = DATA_DIR / "train_series"
TEST_DICOM_DIR = DATA_DIR / "test_series"

print("Path exists: ", os.path.exists(DATA_DIR))

# ---------- Global parameters ----------
TARGET_SIZE = 128        # 128x128 per slice – balances memory and detail
TARGET_SLICES = 30       # fixed number of slices per volume (pad/truncate)
BATCH_SIZE = 64          # larger batch utilises both T4 GPUs
EPOCHS = 50
LR = 1e-4
USE_AMP = True           # mixed precision speeds up training

print("Cell 1 complete.")

Path exists:  True
Cell 1 complete.


In [3]:
# =============================================================================
# Cell 2: Load CSVs and create pseudo-labels from radiology reports
# =============================================================================
train_df = pd.read_csv(TRAIN_CSV)
series_df = pd.read_csv(TRAIN_SERIES_CSV)
label_cols = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
              'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
              'Synovitis', "Baker's", 'Contusion', 'Fracture']

# ---------- Generate pseudo-labels if not already present ----------
# (If you already ran Cell 16, this will be fast because it checks existence)
if not all(f'pseudo_{col}' in train_df.columns for col in label_cols):
    print("Generating pseudo-labels from reports... (this may take a minute)")
    # Keyword lists (same as before) – you could upgrade to a small NLP model later
    LABEL_KEYWORDS = {
        'ACL': ['acl', 'anterior cruciate', 'ligamento cruzado anterior', 'vorderes kreuzband'],
        'MCL': ['mcl', 'medial collateral', 'ligamento colateral medial', 'inneres seitenband'],
        'Medial Meniscus': ['menisco medial', 'medial meniscus', 'innenmeniskus', 'ménisque médial'],
        'Lateral Meniscus': ['menisco lateral', 'lateral meniscus', 'außenmeniskus', 'ménisque latéral'],
        'Medial OA': ['artrosis medial', 'osteoarthritis medial', 'gonarthrose medial', 'medial compartment osteoarthritis'],
        'Lateral OA': ['artrosis lateral', 'osteoarthritis lateral', 'gonarthrose lateral'],
        'PF OA': ['patellofemoral', 'artrosis patelofemoral', 'patellofemorale arthrose', 'pf oa'],
        'Effusion': ['derrame', 'effusion', 'erguss', 'épanchement', 'joint effusion'],
        'Synovitis': ['sinovitis', 'synovitis', 'synovialitis', 'synovite'],
        "Baker's": ['baker', 'quiste de baker', 'bakerzyste', 'kyste de baker'],
        'Contusion': ['contusión', 'contusion', 'bone bruise', 'prellung', 'bone marrow edema'],
        'Fracture': ['fractura', 'fracture', 'fraktur', 'fissure', 'fract']
    }
    import re
    def extract_labels(text):
        if not isinstance(text, str):
            return {col: 0 for col in label_cols}
        text_lower = text.lower()
        pred = {}
        for col, keywords in LABEL_KEYWORDS.items():
            pred[col] = int(any(re.search(r'\b' + re.escape(kw) + r'\b', text_lower) for kw in keywords))
        return pred

    # Apply extraction
    pseudo_dict = train_df['Report'].apply(extract_labels)
    pseudo_df = pd.DataFrame(pseudo_dict.tolist(), index=train_df.index)
    pseudo_df.columns = [f'pseudo_{c}' for c in label_cols]
    train_df = pd.concat([train_df, pseudo_df], axis=1)
    print("Pseudo-labels added.")
else:
    print("Pseudo-labels already exist.")

# ---------- Always use pseudo-labels as targets ----------
print("Using pseudo-labels as training targets.")
print("Positive counts (pseudo):")
print(train_df[[f'pseudo_{c}' for c in label_cols]].sum().sort_values(ascending=False))

# Save for later (optional)
train_df.to_csv("train_with_pseudo.csv", index=False)
print("Cell 2 complete.")

Generating pseudo-labels from reports... (this may take a minute)
Pseudo-labels added.
Using pseudo-labels as training targets.
Positive counts (pseudo):
pseudo_Effusion            2115
pseudo_Lateral Meniscus    1710
pseudo_Medial Meniscus     1658
pseudo_MCL                 1630
pseudo_ACL                 1585
pseudo_Baker's             1219
pseudo_PF OA               1141
pseudo_Contusion            874
pseudo_Fracture             725
pseudo_Synovitis            419
pseudo_Medial OA             13
pseudo_Lateral OA             1
dtype: int64
Cell 2 complete.


In [13]:
# =============================================================================
# Cell 3: Fast parallel preloading with multiprocessing (global map)
# =============================================================================
import multiprocessing as mp

# Global variables for workers
GLOBAL_STUDY_SERIES_MAP = None
GLOBAL_DICOM_DIR = None
GLOBAL_TARGET_SIZE = None
GLOBAL_TARGET_SLICES = None

def init_worker(study_series_map, dicom_dir, target_size, target_slices):
    """Initialize global variables in each worker process."""
    global GLOBAL_STUDY_SERIES_MAP, GLOBAL_DICOM_DIR, GLOBAL_TARGET_SIZE, GLOBAL_TARGET_SLICES
    GLOBAL_STUDY_SERIES_MAP = study_series_map
    GLOBAL_DICOM_DIR = Path(dicom_dir)
    GLOBAL_TARGET_SIZE = target_size
    GLOBAL_TARGET_SLICES = target_slices

def load_one_volume_worker(study_id):
    """Worker function to load a single volume."""
    series_list = GLOBAL_STUDY_SERIES_MAP.get(study_id, [])
    for series_uid in series_list:
        series_path = GLOBAL_DICOM_DIR / study_id / series_uid
        raw = load_series_volume(series_path)
        if raw is not None:
            proc = preprocess_volume(raw, GLOBAL_TARGET_SIZE, GLOBAL_TARGET_SLICES)
            if proc is not None:
                return proc.astype(np.float16)
    # Fallback zeros
    return np.zeros((GLOBAL_TARGET_SLICES, GLOBAL_TARGET_SIZE, GLOBAL_TARGET_SIZE), dtype=np.float16)

class KneeMRIDataset(Dataset):
    def __init__(self, df, dicom_dir, series_df, label_cols,
                 target_size=128, target_slices=30, preload=True, n_workers=4):
        self.df = df.reset_index(drop=True)
        self.dicom_dir = Path(dicom_dir)
        self.target_cols = [f'pseudo_{col}' for col in label_cols]
        self.target_size = target_size
        self.target_slices = target_slices

        # Build series priority map
        self.study_series_map = {}
        for study_id, group in series_df.groupby('StudyInstanceUID'):
            def priority(row):
                score = 0
                if row['Anatomical_Plane'] == 'Sagittal':
                    score += 2
                if row['Fluid_Sensitive'] == 1:
                    score += 2
                return score
            group = group.copy()
            group['priority'] = group.apply(priority, axis=1)
            group = group.sort_values('priority', ascending=False)
            self.study_series_map[study_id] = group['SeriesInstanceUID'].tolist()

        self.volumes = None
        if preload:
            dtype = np.float16
            shape = (len(self.df), target_slices, target_size, target_size)
            self.volumes = np.zeros(shape, dtype=dtype)
            print(f"Allocating {shape} with {dtype.__name__} -> {self.volumes.nbytes / 1e9:.2f} GB")

            # Prepare for parallel loading
            study_ids = self.df['StudyInstanceUID'].tolist()
            # Use multiprocessing with initializer
            with mp.Pool(n_workers, initializer=init_worker,
                         initargs=(self.study_series_map, self.dicom_dir, target_size, target_slices)) as pool:
                results = list(tqdm(pool.imap(load_one_volume_worker, study_ids),
                                    total=len(study_ids), desc="Loading volumes"))
            # Fill array
            for idx, vol in enumerate(results):
                self.volumes[idx] = vol
            print("Preloading complete.")

    def _load_one(self, study_id):
        # Use the same logic as worker, but for single load (if preload=False)
        series_list = self.study_series_map.get(study_id, [])
        for series_uid in series_list:
            series_path = self.dicom_dir / study_id / series_uid
            raw = load_series_volume(series_path)
            if raw is not None:
                proc = preprocess_volume(raw, self.target_size, self.target_slices)
                if proc is not None:
                    return proc
        return None

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        if self.volumes is not None:
            vol = self.volumes[idx].astype(np.float32)
        else:
            study_id = self.df.iloc[idx]['StudyInstanceUID']
            vol = self._load_one(study_id)
            if vol is None:
                vol = np.zeros((self.target_slices, self.target_size, self.target_size), dtype=np.float32)
        vol_tensor = torch.tensor(vol, dtype=torch.float32).unsqueeze(0)
        labels = torch.tensor(self.df.iloc[idx][self.target_cols].values.astype(np.float32), dtype=torch.float32)
        labels = torch.nan_to_num(labels, nan=0.0)
        return {'volume': vol_tensor, 'labels': labels}

# Instantiate
ds = KneeMRIDataset(train_df, TRAIN_DICOM_DIR, series_df, label_cols,
                    target_size=TARGET_SIZE, target_slices=TARGET_SLICES,
                    preload=True, n_workers=4)
print("Cell 3 complete.")

Allocating (4407, 30, 128, 128) with float16 -> 4.33 GB


Loading volumes: 100%|██████████| 4407/4407 [11:25<00:00,  6.43it/s]


Preloading complete.
Cell 3 complete.


In [5]:
# =============================================================================
# Cell 4: 3D CNN model (3D ResNet18-like) for full volume classification
# =============================================================================
class Simple3DCNN(nn.Module):
    """A lightweight 3D CNN that processes the whole volume at once."""
    def __init__(self, in_channels=1, num_classes=12):
        super().__init__()
        # Conv3D layers with batch norm and pooling
        self.conv1 = nn.Conv3d(in_channels, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm3d(32)
        self.pool1 = nn.MaxPool3d(2)  # 30→15

        self.conv2 = nn.Conv3d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm3d(64)
        self.pool2 = nn.MaxPool3d(2)  # 15→7

        self.conv3 = nn.Conv3d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm3d(128)
        self.pool3 = nn.MaxPool3d(2)  # 7→3

        self.conv4 = nn.Conv3d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm3d(256)
        self.pool4 = nn.MaxPool3d(2)  # 3→1 (if kernel size not divisible, adaptive pooling)

        # Adaptive pooling to handle variable dimensions
        self.global_pool = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        # x: (batch, 1, slices, H, W)
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool1(x)
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool2(x)
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.pool3(x)
        x = F.relu(self.bn4(self.conv4(x)))
        x = self.pool4(x)
        x = self.global_pool(x)          # (batch, 256, 1, 1, 1)
        x = x.view(x.size(0), -1)        # (batch, 256)
        return self.fc(x)

# For better performance, you could use a pretrained 3D ResNet from MONAI.
# But this simple model is faster to train and still captures 3D context.

print("Cell 4 complete – model defined.")

Cell 4 complete – model defined.


In [ ]:
# =============================================================================
# Cell 5: Training loop with mixed precision, GPU monitoring, and per-epoch AUC
# =============================================================================
EPOCHS = 50

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gpu_count = torch.cuda.device_count()
print(f"Using {gpu_count} GPU(s)")

# Data splits
train_idx, val_idx = train_test_split(range(len(ds)), test_size=0.2, random_state=42)
train_dataset = torch.utils.data.Subset(ds, train_idx)
val_dataset = torch.utils.data.Subset(ds, val_idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=False, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, pin_memory=False)

# Model, loss, optimizer
model = Simple3DCNN(in_channels=1, num_classes=12).to(device)
if gpu_count > 1:
    model = nn.DataParallel(model)
    print("DataParallel enabled")

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scaler = GradScaler(enabled=USE_AMP)

# AUC helper
def compute_auc(labels, preds, label_names):
    auc_dict = {}
    valid = []
    for i, name in enumerate(label_names):
        if labels[:, i].sum() > 0 and (labels[:, i] == 0).sum() > 0:
            try:
                auc = roc_auc_score(labels[:, i].cpu().numpy(), preds[:, i].cpu().numpy())
                auc_dict[name] = auc
                valid.append(auc)
            except:
                auc_dict[name] = float('nan')
        else:
            auc_dict[name] = float('nan')
    macro = np.nanmean(valid) if valid else 0.0
    return auc_dict, macro

def fmt_auc(auc_dict):
    return " | ".join([f"{k}={v:.3f}" if not np.isnan(v) else f"{k}=nan" for k,v in auc_dict.items()])

best_val_auc = 0.0
for epoch in range(1, EPOCHS+1):
    model.train()
    train_loss = 0.0
    start_time = time.time()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]", leave=False)
    for batch in pbar:
        volumes = batch['volume'].to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)
        optimizer.zero_grad()
        with autocast(enabled=USE_AMP):
            logits = model(volumes)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item() * volumes.size(0)
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    avg_train_loss = train_loss / len(train_dataset)

    # Validation
    model.eval()
    val_loss = 0.0
    all_labels, all_preds = [], []
    pbar = tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [Valid]", leave=False)
    with torch.no_grad():
        for batch in pbar:
            volumes = batch['volume'].to(device, non_blocking=True)
            labels = batch['labels'].to(device, non_blocking=True)
            with autocast(enabled=USE_AMP):
                logits = model(volumes)
                loss = criterion(logits, labels)
            val_loss += loss.item() * volumes.size(0)
            all_labels.append(labels.cpu())
            all_preds.append(torch.sigmoid(logits).cpu())
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    avg_val_loss = val_loss / len(val_dataset)
    all_labels = torch.cat(all_labels, dim=0)
    all_preds = torch.cat(all_preds, dim=0)
    per_label_auc, macro_auc = compute_auc(all_labels, all_preds, label_cols)

    # Memory stats
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated(device) / 1e9
        reserv = torch.cuda.memory_reserved(device) / 1e9
        mem_str = f"GPU Mem: {alloc:.2f}/{reserv:.2f} GB"
    else:
        mem_str = ""

    elapsed = time.time() - start_time
    print(f"\n{'='*80}")
    print(f"Epoch {epoch:2d}/{EPOCHS} | Time: {elapsed:.1f}s | {mem_str}")
    print(f"  Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Macro AUC: {macro_auc:.4f}")
    print(f"  Per-label AUC: {fmt_auc(per_label_auc)}")

    if macro_auc > best_val_auc:
        best_val_auc = macro_auc
        state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
        torch.save(state, "best_model.pth")
        print(f"  ✅ New best model saved (AUC: {best_val_auc:.4f})")
    else:
        print(f"  Best so far: {best_val_auc:.4f} (epoch {epoch})")
    print('='*80 + "\n")

    # Clean up GPU cache
    torch.cuda.empty_cache()
    gc.collect()

print(f"\n🎉 Training complete. Best validation AUC: {best_val_auc:.4f}")

Using 2 GPU(s)
DataParallel enabled



Epoch  1/50 | Time: 36.4s | GPU Mem: 0.15/6.50 GB
  Train Loss: 0.4819 | Val Loss: 0.4609 | Macro AUC: 0.6464
  Per-label AUC: ACL=0.719 | MCL=0.711 | Medial Meniscus=0.658 | Lateral Meniscus=0.639 | Medial OA=0.388 | Lateral OA=nan | PF OA=0.647 | Effusion=0.614 | Synovitis=0.679 | Baker's=0.614 | Contusion=0.734 | Fracture=0.708
  ✅ New best model saved (AUC: 0.6464)




Epoch  2/50 | Time: 34.9s | GPU Mem: 0.14/6.11 GB
  Train Loss: 0.4478 | Val Loss: 0.4423 | Macro AUC: 0.6949
  Per-label AUC: ACL=0.749 | MCL=0.731 | Medial Meniscus=0.687 | Lateral Meniscus=0.672 | Medial OA=0.678 | Lateral OA=nan | PF OA=0.693 | Effusion=0.643 | Synovitis=0.688 | Baker's=0.648 | Contusion=0.737 | Fracture=0.719
  ✅ New best model saved (AUC: 0.6949)



Epoch 3/50 [Train]:  69%|██████▉   | 38/55 [00:21<00:09,  1.77it/s, loss=0.4265]

In [8]:
# =============================================================================
# Cell 6: Load best model, predict on test set, and create submission.csv
# =============================================================================
test_df = pd.read_csv(TEST_CSV)
test_series_df = pd.read_csv(TEST_SERIES_CSV)

# Use the same dataset class for test (no labels needed)
class TestKneeMRIDataset(KneeMRIDataset):
    def __init__(self, df, dicom_dir, series_df, **kwargs):
        super().__init__(df, dicom_dir, series_df, label_cols, **kwargs)
    def __getitem__(self, idx):
        vol = self.volumes[idx] if self.volumes is not None else self._load_one(self.df.iloc[idx]['StudyInstanceUID'])
        vol_tensor = torch.tensor(vol, dtype=torch.float32).unsqueeze(0)
        return {'volume': vol_tensor, 'study_id': self.df.iloc[idx]['StudyInstanceUID']}

# Build test dataset with preloading
test_ds = TestKneeMRIDataset(test_df, TEST_DICOM_DIR, test_series_df,
                             target_size=TARGET_SIZE, target_slices=TARGET_SLICES, preload=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# ---------- Load best model correctly ----------
# 1. Instantiate the model (unwrapped)
model = Simple3DCNN(in_channels=1, num_classes=12).to(device)

# 2. Load the saved state_dict (keys have no "module." prefix)
model.load_state_dict(torch.load("best_model.pth", map_location=device))

# 3. Now wrap with DataParallel if multiple GPUs are available
if gpu_count > 1:
    model = nn.DataParallel(model)
    print("DataParallel enabled for inference")

model.eval()

# Predict
all_probs = []
all_ids = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting"):
        volumes = batch['volume'].to(device)
        with autocast(enabled=USE_AMP):
            logits = model(volumes)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_ids.extend(batch['study_id'])
all_probs = np.concatenate(all_probs, axis=0)

# Create submission
submission = pd.DataFrame(all_probs, columns=label_cols)
submission.insert(0, 'StudyInstanceUID', all_ids)
submission.to_csv("submission.csv", index=False)
print(f"✅ Submission saved with {len(submission)} rows.")
print(submission.head())

Preloading 3 volumes (128x128)...


100%|██████████| 3/3 [00:00<00:00,  7.96it/s]


Preloading complete. Estimated RAM usage: 0.01 GB
DataParallel enabled for inference


Predicting: 100%|██████████| 1/1 [00:00<00:00, 12.82it/s]

✅ Submission saved with 3 rows.
                                    StudyInstanceUID       ACL       MCL  \
0  1.2.826.0.1.3680043.8.498.10047035057544427318...  0.059113  0.276367   
1  1.2.826.0.1.3680043.8.498.10062861783145312629...  0.692383  0.186523   
2  1.2.826.0.1.3680043.8.498.10067514707072572280...  0.187622  0.109924   

   Medial Meniscus  Lateral Meniscus  Medial OA  Lateral OA     PF OA  \
0         0.457520          0.297363   0.000947    0.000385  0.120850   
1         0.551758          0.510254   0.005959    0.001107  0.084167   
2         0.080933          0.071716   0.001796    0.001184  0.235962   

   Effusion  Synovitis   Baker's  Contusion  Fracture  
0  0.736328   0.006168  0.168091   0.188354  0.129639  
1  0.818848   0.143555  0.553711   0.571777  0.164429  
2  0.615723   0.015541  0.388672   0.301270  0.125977  
